# Test `Fast_VTON_full.pt` trên Kaggle GPU T4 x2

Notebook test model virtual try-on `Fast_VTON_full.pt` (Stage 1) đã huấn luyện xong, thiết kế **cho Kaggle Notebook với GPU (khuyên dùng T4 x2, mỗi con 15 GB)**.

**Luồng:** kiểm tra GPU → cài dependencies (Fast-VTON + dự án test) → tải model vào `models/` bằng `gdown` → kiểm tra nhanh pipeline → chạy Gradio public link để up ảnh người + ảnh quần áo.

> **Quan trọng:** trong Kaggle, chọn **Accelerator → GPU T4 x2** và bật **Internet** trong Settings. Internet cần cho `gdown`, lần tải đầu tiên của Segformer/scheduler, và Gradio tạo public link.
>
> **Lưu ý CUDA:** PyTorch bản CUDA 12 hiện tại chỉ hỗ trợ GPU có compute capability **>= 7.0** (T4 = sm_75, V100 = sm_70). **P100 (sm_60) không chạy được** — hãy dùng T4. VRAM 15 GB x2 dư sức cho bundle fp16 (~3.2 GB). Bundle là fp16 nên chạy tốt trên T4 (không cần bf16). Phải `pip uninstall -y peft` (xung đột với diffusers 0.22).

In [ ]:
import os
import torch

IS_KAGGLE = os.path.exists('/kaggle/input') or ('KAGGLE_CONTAINER_NAME' in os.environ)
if not IS_KAGGLE:
    raise RuntimeError('Notebook này chỉ hỗ trợ Kaggle Notebook có GPU.')
if not torch.cuda.is_available():
    raise RuntimeError('Chưa bật GPU. Vào Kaggle Settings → Accelerator và chọn GPU (T4 x2).')

# PyTorch CUDA 12 chỉ hỗ trợ GPU compute capability >= 7.0 (T4 = sm_75, V100 = sm_70).
# P100 (sm_60) sẽ báo lỗi 'not compatible' khi chạy → không dùng.
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
cc_major = cc[0]
print('GPU0:', gpu_name, '| compute capability:', cc)
if cc_major < 7:
    raise RuntimeError(f'GPU {gpu_name} (sm_{cc_major}0) quá cũ. Dùng T4/V100 trở lên (sm_70+).')

ngpu = torch.cuda.device_count()
print('Số GPU khả dụng:', ngpu)
for i in range(ngpu):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  [{i}]', torch.cuda.get_device_name(i), f'{mem:.1f} GB')
print('Kaggle GPU đã xác nhận.')

WORKDIR = os.getcwd()
PARENT = os.path.dirname(WORKDIR)
FAST_VTON_DIR = os.path.join(PARENT, 'Fast-VTON')
TEST_VTON_DIR = os.path.join(PARENT, 'Test_FAST_VTON_Stage_1')
print('CWD hiện tại:', WORKDIR)
print('Fast-VTON   :', FAST_VTON_DIR)
print('Test-VTON   :', TEST_VTON_DIR)

In [ ]:
import subprocess
import sys

def _clone_or_pull(repo_url, dest):
    # Chưa có -> clone; đã có -> git pull để luôn chạy code mới nhất trên repo
    # (tránh tình trạng sửa code trên GitHub nhưng bản clone cũ trong Kaggle không theo kịp).
    if os.path.isdir(os.path.join(dest, '.git')):
        try:
            subprocess.run(['git', '-C', dest, 'pull', '--ff-only'],
                            check=True, capture_output=True)
            print('git pull OK :', dest)
        except subprocess.CalledProcessError as e:
            print('git pull thất bại (bỏ qua):', e)
    else:
        subprocess.run(['git', 'clone', '-q', repo_url, dest], check=True)
        print('git clone OK:', dest)

_clone_or_pull('https://github.com/hoangtung386/Fast-VTON.git', FAST_VTON_DIR)
_clone_or_pull('https://github.com/hoangtung386/Test_FAST_VTON_Stage_1.git', TEST_VTON_DIR)

# Gốc dự án test là nơi chứa models/ và src/fast_vton.
WORKDIR = TEST_VTON_DIR

# Đảm bảo import được codebase bất kể editable install có thành công hay không.
# Dự án dùng src-layout (package ở src/fast_vton) nên PHẢI thêm .../src vào sys.path
# để 'import fast_vton' tìm thấy; còn Fast-VTON export package 'src' từ FAST_VTON_DIR.
for _d in (os.path.join(WORKDIR, 'src'), FAST_VTON_DIR):
    if _d not in sys.path:
        sys.path.insert(0, _d)

if not os.path.isfile(os.path.join(WORKDIR, 'src', 'fast_vton', 'pipeline.py')):
    raise RuntimeError(
        f'Không tìm thấy src/fast_vton/pipeline.py tại {WORKDIR}. '
        f'Clone dự án test thất bại? Kiểm tra Internet và quyền truy cập repo.'
    )
print('Codebase dự án sẵn sàng tại:', WORKDIR)
print('sys.path đã thêm:', os.path.join(WORKDIR, 'src'), '|', FAST_VTON_DIR)

## Bước 1 — Cài đặt môi trường

**Sửa lỗi `grad_and_value` (quan trọng):** `diffusers==0.22.0` chỉ tương thích với `torch<=2.3`.
Trên Kaggle, bản torch mặc định thường là **2.4+** → gây lỗi
`module 'torch._functorch.eager_transforms' has no attribute 'grad_and_value'`
ngay khi import `DDPMScheduler`. Vì vậy BẮT BUỘC ép cài `torch==2.2.1` (từ index CUDA 12.1
của PyTorch) và **xác nhận version** sau khi cài — bản cài cũ dùng `-qqq` đã làm lệnh
downgrade torch thất bại trong im lặng, để lại torch mới gây crash.

Phiên bản chuẩn theo Fast-VTON: `diffusers==0.22.0`, `transformers==4.37.2`, `torch==2.2.1`.
Cuối cùng **gỡ `peft`** để diffusers không bật PEFT backend gây vỡ giữa lượt UNet.

> Chạy trên **T4 x2**: torch 2.2.1 (cu121) chạy được trên driver CUDA 12 (forward compat)
> và hỗ trợ compute capability sm_75 của T4.

In [ ]:
import subprocess

def _run(cmd):
    print(">>>", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if len(out) > 4000:  # chỉ in phần cuối để tránh tràn output
        out = out[-4000:]
    print(out)
    if r.returncode != 0:
        raise RuntimeError(f"Lệnh thất bại (exit {r.returncode}): {cmd}")
    return r

# 1) Torch 2.2.1 (cu121) - ép cài lại để GHI ĐÈ torch mặc định của Kaggle (thường >= 2.4
#    và gây lỗi grad_and_value với diffusers 0.22). Dùng index chính thức của PyTorch.
_run(
    "pip install --no-cache-dir --force-reinstall "
    "--extra-index-url https://download.pytorch.org/whl/cu121 "
    "torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1"
)

# 2) Fast-VTON (package swiftedit). Truyền index PyTorch để không kéo torch mới.
_run(f"pip install --extra-index-url https://download.pytorch.org/whl/cu121 -e {FAST_VTON_DIR}[vton]")

# 3) numpy < 2 (diffusers 0.22 yêu cầu)
_run("pip install numpy==1.26.4")

# 4) Gỡ peft (xung đột với diffusers 0.22)
_run("pip uninstall -y peft")

# 5) Dự án test
_run(f"pip install --extra-index-url https://download.pytorch.org/whl/cu121 -e {WORKDIR}")

# 6) Xác nhận version - báo lỗi rõ ràng nếu torch/diffusers không như ý.
import torch, diffusers
print("torch     :", torch.__version__)
print("diffusers :", diffusers.__version__)
assert torch.__version__.startswith("2.2.1"), (
    f"torch phải là 2.2.1 để chạy diffusers 0.22, nhưng có {torch.__version__}. "
    "Cài đặt torch thất bại - xem output lệnh pip ở trên."
)
assert diffusers.__version__ == "0.22.0", (
    f"diffusers phải là 0.22.0, nhưng có {diffusers.__version__}."
)
print("Môi trường đã sẵn sàng.")


## Bước 2 — Tải model `Fast_VTON_full.pt` từ Google Drive

Tải trực tiếp file public từ Google Drive bằng `gdown`. File sẽ được đặt vào `models/Fast_VTON_full.pt` (vị trí mặc định `Config().bundle_path`).

In [ ]:
import os

MODEL_DIR = os.path.join(WORKDIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'Fast_VTON_full.pt')

GDRIVE_MODEL_URL = 'https://drive.google.com/file/d/14CX8n1co5riWvX2KHVFI6Iud7Vi8MmJJ/view?usp=sharing'

if not os.path.exists(MODEL_PATH):
    !pip install -q gdown
    !gdown --fuzzy {GDRIVE_MODEL_URL} -O {MODEL_PATH}
    if not os.path.isfile(MODEL_PATH) or os.path.getsize(MODEL_PATH) == 0:
        raise RuntimeError('Không tải được checkpoint. Kiểm tra Kaggle Internet và quyền public của link Google Drive.')
    print('đã chuẩn bị', MODEL_PATH)
else:
    print('model đã có sẵn tại', MODEL_PATH)

## Bước 3 — Kiểm tra nhanh pipeline (bắt lỗi môi trường sớm)

Chạy thử trên ảnh synthetic để xác nhận: bundle load được, DINOv2/CLIP/VAE/inversion hoạt động, mask + forward chạy qua. Ảnh rác nên kết quả chỉ để verify không crash.

In [ ]:
import os
import torch
from PIL import Image
from fast_vton.pipeline import FastVTONPipeline

bundle = os.path.join(WORKDIR, 'models', 'Fast_VTON_full.pt')
pred = FastVTONPipeline(bundle_path=bundle, device='cuda')
person = Image.new('RGB', (384, 512), (220, 200, 180))
garment = Image.new('RGB', (224, 224), (30, 90, 200))
agnostic = pred.build_agnostic(person)
out = pred.try_on(person, agnostic, garment)
print('bundle step :', pred.bundle.manifest.step)
print('output shape:', tuple(out.shape))

## Bước 4 — Chạy Gradio để test thật

Mở public link `gradio.live` hiện ra, up **ảnh người mẫu** + **ảnh quần áo**, bấm *Thử đồ*. Ảnh agnostic tự sinh (human parsing). Để tắt auto-agnostic, bỏ tick và up sẵn ảnh agnostic.

In [ ]:
%cd {WORKDIR}
from fast_vton.app import build_demo
build_demo().launch(share=True, debug=False)

## (Tuỳ chọn) Chạy qua CLI thay vì Gradio

Đặt ảnh test vào `data/` rồi chạy. Hữu ích khi chạy batch hoặc không cần UI.

In [ ]:
print('CLI test:')
print('  python -m fast_vton.cli --person data/nguoi.jpg --garment data/ao.jpg --output outputs/result.png')
print('Tắt auto-agnostic nếu có sẵn ảnh agnostic:')
print('  python -m fast_vton.cli --person data/nguoi.jpg --garment data/ao.jpg --agnostic data/agnostic.jpg --no-auto-agnostic --output outputs/result.png')